# LeetCode 1753 — Maximum Score From Removing Stones bằng Python

Link đề: https://leetcode.com/problems/maximum-score-from-removing-stones/ · Độ khó: **Medium** · Chủ đề: Math, Greedy, Heap (Priority Queue)

Skeleton LeetCode:

```python
class Solution:
    def maximumScore(self, a: int, b: int, c: int) -> int:
        ...
```

## 1. Mục tiêu học

1. Mô hình hóa trò chơi thành **trạng thái** `(a, b, c)` và viết vét cạn có **ghi nhớ** (memoization) để có đáp án chuẩn.
2. Hiểu chiến lược tham lam **"luôn lấy từ hai đống lớn nhất"** và cài bằng **heap** (`heapq`) hoặc sắp xếp lại mỗi bước.
3. Rút ra **công thức đóng** `O(1)`: nếu đống lớn nhất `≥` tổng hai đống còn lại thì đáp án là tổng hai đống nhỏ; ngược lại là `⌊(a+b+c)/2⌋`.
4. Chứng minh công thức bằng hai **cận trên** và một **chiến lược đạt cận**.
5. Kiểm thử đối chiếu mọi cách trên lưới giá trị nhỏ và ngẫu nhiên có seed cố định.

## 2. Đề bài & ví dụ

Có ba đống sỏi kích thước `a`, `b`, `c`. Mỗi lượt, chọn **hai đống khác nhau, đều không rỗng**, lấy mỗi đống **một** viên và được **+1 điểm**. Trò chơi dừng khi còn **ít hơn hai** đống không rỗng. Trả về **điểm tối đa**.

**Ví dụ 1:** `a = 2, b = 4, c = 6` → `6`. Ví dụ một chuỗi tối ưu: (2,4,6) → (1,4,5) → (0,4,4) → (0,3,3) → (0,2,2) → (0,1,1) → (0,0,0).

**Ví dụ 2:** `a = 4, b = 4, c = 6` → `7`. (4,4,6) → (3,3,6) → (2,3,5) → (1,3,4) → (0,3,3) → … → (0,0,0).

**Ví dụ 3:** `a = 1, b = 8, c = 8` → `8`. Lấy từ đống 2 và 3 đúng 8 lần.

**Ràng buộc:** `1 ≤ a, b, c ≤ 10⁵`.

In [1]:
import heapq
import random
import time
from functools import lru_cache

examples = [((2, 4, 6), 6), ((4, 4, 6), 7), ((1, 8, 8), 8)]
for (a, b, c), exp in examples:
    print(f"a={a}, b={b}, c={c} -> kỳ vọng {exp}")

a=2, b=4, c=6 -> kỳ vọng 6
a=4, b=4, c=6 -> kỳ vọng 7
a=1, b=8, c=8 -> kỳ vọng 8


## 3. Kiến thức nền

### 3.1 Trạng thái và chuyển trạng thái

Mỗi lượt chỉ có 3 lựa chọn: lấy từ cặp `(1,2)`, `(1,3)` hoặc `(2,3)` — nếu cả hai đống đều `> 0`. Từ `(a, b, c)` ta đi tới tối đa 3 trạng thái mới. Điểm = số lượt đi được.

In [2]:
def moves(state):
    a, b, c = state
    nxt = []
    if a and b: nxt.append((a - 1, b - 1, c))
    if a and c: nxt.append((a - 1, b, c - 1))
    if b and c: nxt.append((a, b - 1, c - 1))
    return nxt

print(moves((2, 4, 6)))
print(moves((0, 0, 5)), "-> rỗng: trò chơi kết thúc")

[(1, 3, 6), (1, 4, 5), (2, 3, 5)]
[] -> rỗng: trò chơi kết thúc


### 3.2 `functools.lru_cache` — ghi nhớ kết quả hàm

Nếu một hàm đệ quy được gọi lại nhiều lần với cùng tham số, `@lru_cache(maxsize=None)` lưu kết quả để lần sau trả ngay. Biến vét cạn từ hàm mũ thành "số trạng thái × chi phí mỗi trạng thái".

In [3]:
@lru_cache(maxsize=None)
def fib(n):
    return n if n < 2 else fib(n - 1) + fib(n - 2)
print(fib(80), fib.cache_info())

23416728348467685 CacheInfo(hits=78, misses=81, maxsize=None, currsize=81)


### 3.3 `heapq` — heap nhỏ nhất, mẹo đổi dấu để có heap lớn nhất

`heapq` của Python là **min-heap**. Muốn lấy phần tử **lớn nhất**, ta đẩy `-x` vào và lấy `-heappop(...)`.

In [4]:
h = []
for x in [2, 4, 6]:
    heapq.heappush(h, -x)
print("lớn nhất:", -heapq.heappop(h), "| lớn nhì:", -heapq.heappop(h), "| còn:", [-v for v in h])

lớn nhất: 6 | lớn nhì: 4 | còn: [2]


### 3.4 Sắp xếp ba số và phép chia nguyên `//`

`a, b, c = sorted((a, b, c))` cho `a ≤ b ≤ c`. `(a + b + c) // 2` là phần nguyên của nửa tổng.

In [5]:
a, b, c = sorted((6, 2, 4))
print(a, b, c, "| nửa tổng:", (a + b + c) // 2, "| 15 // 2 =", 15 // 2)

2 4 6 | nửa tổng: 6 | 15 // 2 = 7


## 4. Ý tưởng nền tảng / trực giác

Mỗi lượt lấy **2 viên** ⇒ sau `k` lượt đã mất `2k` viên ⇒ `k ≤ ⌊(a+b+c)/2⌋`. Đây là **cận trên thứ nhất**.

Nhưng mỗi lượt phải có **ít nhất một viên ngoài đống lớn nhất** (`c`)? Đúng: một lượt lấy từ hai đống khác nhau nên tối đa một viên thuộc đống `c`, viên còn lại thuộc `a` hoặc `b`. Vậy số lượt `≤ a + b`. Đây là **cận trên thứ hai**.

Tính tay:
- `(1, 8, 8)`: cận 1 = `⌊17/2⌋ = 8`, cận 2 = `1 + 8 = 9` ⇒ tối đa 8 ✓.
- `(1, 2, 10)`: cận 1 = `6`, cận 2 = `3` ⇒ tối đa 3 (đống 10 quá lớn, không có đủ "bạn" để ghép).
- `(4, 4, 6)`: cận 1 = `7`, cận 2 = `8` ⇒ tối đa 7 ✓.

Giả thuyết: **đáp án = min(⌊(a+b+c)/2⌋, a + b)** với `c` lớn nhất. Ta sẽ kiểm chứng bằng vét cạn.

In [6]:
for t in [(1, 8, 8), (1, 2, 10), (4, 4, 6), (2, 4, 6)]:
    a, b, c = sorted(t)
    print(t, "cận 1 =", (a + b + c) // 2, "| cận 2 =", a + b, "| min =", min((a + b + c) // 2, a + b))

(1, 8, 8) cận 1 = 8 | cận 2 = 9 | min = 8
(1, 2, 10) cận 1 = 6 | cận 2 = 3 | min = 3
(4, 4, 6) cận 1 = 7 | cận 2 = 8 | min = 7
(2, 4, 6) cận 1 = 6 | cận 2 = 6 | min = 6


## 5. Cách 1 — Brute Force / baseline: vét cạn mọi chuỗi nước đi có ghi nhớ

**Ý tưởng:** `best(a, b, c)` = 0 nếu không còn nước đi; ngược lại `1 + max(best(trạng thái kế tiếp))`. Chuẩn hóa trạng thái bằng `sorted` để giảm số trạng thái (thứ tự các đống không quan trọng).

In [7]:
def score_bruteforce(a, b, c):
    @lru_cache(maxsize=None)
    def best(x, y, z):
        res = 0
        for nx in moves((x, y, z)):
            res = max(res, 1 + best(*sorted(nx)))
        return res
    return best(*sorted((a, b, c)))

for (a, b, c), exp in examples:
    got = score_bruteforce(a, b, c)
    print(got, "kỳ vọng", exp); assert got == exp

6 kỳ vọng 6
7 kỳ vọng 7
8 kỳ vọng 8


**Độ phức tạp:** số trạng thái tới `O(a·b·c)` (sau chuẩn hóa ít hơn nhưng vẫn bậc ba), mỗi trạng thái `O(1)`. Với `10⁵` thì `10¹⁵` trạng thái — **không thể**. Đệ quy sâu tới `(a+b+c)/2` cũng vượt giới hạn đệ quy của Python.

**Nút thắt:** ta thử **mọi** lựa chọn trong khi có thể chỉ cần một lựa chọn "khôn ngoan" ở mỗi bước.

## 6. Ý tưởng thuật toán / bước đột phá

**Insight tham lam:** mỗi lượt, lấy từ **hai đống lớn nhất**. Trực giác: giữ các đống "cân bằng" để không đống nào bị cạn sớm làm mất bạn ghép.

**Insight công thức:** sắp `a ≤ b ≤ c`.
- **Nếu** `c ≥ a + b` **thì** đống `c` luôn đủ để ghép với **mọi** viên của `a` và `b` ⇒ đáp án `a + b` (đạt cận 2).
- **Nếu** `c < a + b` **thì** có thể lấy hết (hoặc chỉ dư 1 viên) ⇒ đáp án `⌊(a+b+c)/2⌋` (đạt cận 1).

Gộp lại: `min(a + b, (a + b + c) // 2)`.

**Invariant của tham lam** (khi `c < a + b`): sau mỗi bước lấy từ hai đống lớn nhất, đống lớn nhất mới vẫn `≤` tổng hai đống kia + 1, nên ta không bao giờ rơi vào tình trạng "một đống khổng lồ, hai đống rỗng" trước khi gần hết sỏi.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Mô phỏng tham lam bằng sắp xếp mỗi bước

**Giải thích từng đoạn:**
1. `s = sorted([a, b, c])`.
2. Trong khi đống lớn nhì `s[1] > 0` (tức còn ít nhất hai đống không rỗng): trừ 1 ở `s[1]` và `s[2]`, cộng điểm, sắp lại.

In [8]:
def score_greedy_sort(a, b, c):
    s = sorted([a, b, c])
    ans = 0
    while s[1] > 0:
        s[1] -= 1
        s[2] -= 1
        ans += 1
        s.sort()
    return ans

for (a, b, c), exp in examples:
    assert score_greedy_sort(a, b, c) == exp
print([score_greedy_sort(*t) for t, _ in examples])

[6, 7, 8]


**Code nộp LeetCode (Cách 2):**

In [9]:
class SolutionSort:
    def maximumScore(self, a: int, b: int, c: int) -> int:
        s = sorted([a, b, c])
        ans = 0
        while s[1] > 0:
            s[1] -= 1
            s[2] -= 1
            ans += 1
            s.sort()
        return ans

print([SolutionSort().maximumScore(*t) for t, _ in examples])

[6, 7, 8]


**Độ phức tạp Cách 2:** số vòng = đáp án `≤ 1,5·10⁵`, mỗi vòng sắp 3 phần tử `O(1)` ⇒ Time `O(a+b+c)`, Space `O(1)`.

### 7.2 Cách 3 — Tham lam bằng heap lớn nhất

Tổng quát hơn cho **k đống**: lấy hai đống lớn nhất ra khỏi heap, giảm mỗi đống 1, đẩy lại các đống còn `> 0`.

In [10]:
def score_heap(a, b, c):
    h = [-x for x in (a, b, c) if x > 0]
    heapq.heapify(h)
    ans = 0
    while len(h) >= 2:
        x = -heapq.heappop(h); y = -heapq.heappop(h)
        ans += 1
        if x - 1: heapq.heappush(h, -(x - 1))
        if y - 1: heapq.heappush(h, -(y - 1))
    return ans

print([score_heap(*t) for t, _ in examples])

[6, 7, 8]


**Code nộp LeetCode (Cách 3):**

In [11]:
class SolutionHeap:
    def maximumScore(self, a: int, b: int, c: int) -> int:
        h = [-a, -b, -c]
        heapq.heapify(h)
        ans = 0
        while True:
            x = -heapq.heappop(h)
            y = -heapq.heappop(h)
            if y == 0:                 # đống lớn nhì rỗng => chỉ còn <= 1 đống không rỗng
                return ans
            ans += 1
            heapq.heappush(h, -(x - 1))
            heapq.heappush(h, -(y - 1))

print([SolutionHeap().maximumScore(*t) for t, _ in examples])

[6, 7, 8]


**Độ phức tạp Cách 3:** Time `O((a+b+c) · log k)` với `k = 3` ⇒ `O(a+b+c)`; Space `O(k)`.

### 7.3 Cách 4 — Tham lam "nhảy bước" (gom nhiều lượt một lúc)

Thay vì trừ từng viên, ta xử lý cả **khối** lượt đi một lúc (với `a ≤ b ≤ c`):
1. Nếu `c ≥ a + b`: ghép hết `a` và `b` với `c` ⇒ xong, đáp án `a + b`.
2. Ngược lại: ghép `a` với `c` đúng `k = c − b` lần để hai đống lớn **bằng nhau** (hợp lệ vì `k < a`).
3. Mỗi viên còn lại của `a` ghép luân phiên với hai đống lớn (giữ chênh lệch `≤ 1`).
4. Cuối cùng ghép hai đống lớn với nhau: được `⌊phần còn lại / 2⌋` lượt.

Cách này là cầu nối giữa mô phỏng và công thức:

In [12]:
def score_jump(a, b, c):
    a, b, c = sorted((a, b, c))
    if c >= a + b:
        return a + b
    k = c - b                      # ghép a với c đúng k lần để b == c (k <= a vì c < a + b)
    a -= k; c -= k
    # bây giờ a <= b == c: ghép a viên của a xen kẽ với b, c để giữ b, c cân bằng
    ans = k + a                    # mỗi viên a ghép với b hoặc c
    rest = b + c - a               # tổng còn lại ở hai đống bằng nhau (chênh lệch <= 1)
    return ans + rest // 2

for (a, b, c), exp in examples:
    assert score_jump(a, b, c) == exp
print([score_jump(*t) for t, _ in examples])

[6, 7, 8]


**Độ phức tạp Cách 4:** `O(1)`. Nó giải thích **vì sao** công thức đúng: sau khi cân bằng hai đống lớn, mọi viên đều được ghép, chỉ dư tối đa 1 viên.

### 7.4 Cách 5 — Công thức đóng `O(1)`

In [13]:
def score_formula(a, b, c):
    a, b, c = sorted((a, b, c))
    return min(a + b, (a + b + c) // 2)

print([score_formula(*t) for t, _ in examples])

[6, 7, 8]


**Code nộp LeetCode (Cách 5):** xem mục 9. **Độ phức tạp:** Time `O(1)`, Space `O(1)`.

## 8. TRACE từng bước

Trace Cách 2 trên **Ví dụ 2** `(4, 4, 6)`:

| lượt | trước (đã sắp) | lấy từ | sau | điểm |
|---|---|---|---|---|
| 1 | [4,4,6] | 4 và 6 | [4,3,5] → sắp [3,4,5] | 1 |
| 2 | [3,4,5] | 4 và 5 | [3,3,4] | 2 |
| 3 | [3,3,4] | 3 và 4 | [3,2,3] → [2,3,3] | 3 |
| … | … | … | … | … |
| 7 | [0,1,1] | 1 và 1 | [0,0,0] | **7** |

In [14]:
def trace_greedy(a, b, c):
    s = sorted([a, b, c]); ans = 0
    print("bắt đầu:", s, "| công thức dự đoán:", score_formula(a, b, c))
    while s[1] > 0:
        before = s[:]
        s[1] -= 1; s[2] -= 1; ans += 1
        s.sort()
        print(f"lượt {ans}: {before} -> lấy {before[1]} & {before[2]} -> {s}")
    print("Kết quả:", ans, "| sỏi còn dư:", sum(s))
    return ans

assert trace_greedy(4, 4, 6) == 7

bắt đầu: [4, 4, 6] | công thức dự đoán: 7
lượt 1: [4, 4, 6] -> lấy 4 & 6 -> [3, 4, 5]
lượt 2: [3, 4, 5] -> lấy 4 & 5 -> [3, 3, 4]
lượt 3: [3, 3, 4] -> lấy 3 & 4 -> [2, 3, 3]
lượt 4: [2, 3, 3] -> lấy 3 & 3 -> [2, 2, 2]
lượt 5: [2, 2, 2] -> lấy 2 & 2 -> [1, 1, 2]
lượt 6: [1, 1, 2] -> lấy 1 & 2 -> [0, 1, 1]
lượt 7: [0, 1, 1] -> lấy 1 & 1 -> [0, 0, 0]
Kết quả: 7 | sỏi còn dư: 0


Trace trường hợp **đống lớn áp đảo** `(1, 2, 10)` (ví dụ tự dựng — đối chiếu bằng brute force; nên đối chiếu lại đề gốc):

In [15]:
r = trace_greedy(1, 2, 10)
assert r == score_bruteforce(1, 2, 10) == 3

bắt đầu: [1, 2, 10] | công thức dự đoán: 3
lượt 1: [1, 2, 10] -> lấy 2 & 10 -> [1, 1, 9]
lượt 2: [1, 1, 9] -> lấy 1 & 9 -> [0, 1, 8]
lượt 3: [0, 1, 8] -> lấy 1 & 8 -> [0, 0, 7]
Kết quả: 3 | sỏi còn dư: 7


## 9. Code nộp LeetCode

Bản chọn nộp: **Cách 5** — công thức `O(1)`.

In [16]:
class Solution:
    def maximumScore(self, a: int, b: int, c: int) -> int:
        a, b, c = sorted((a, b, c))
        if c >= a + b:
            return a + b
        return (a + b + c) // 2

for (a, b, c), exp in examples:
    got = Solution().maximumScore(a, b, c)
    print(got, exp); assert got == exp

6 6
7 7
8 8


## 10. Vì sao thuật toán đúng

Giả sử `a ≤ b ≤ c`, gọi `S = a + b + c`, `k` = số lượt.

**Cận trên 1:** mỗi lượt bỏ đi 2 viên ⇒ `2k ≤ S` ⇒ `k ≤ ⌊S/2⌋`.

**Cận trên 2:** mỗi lượt lấy từ hai đống **khác nhau** ⇒ tối đa 1 viên từ đống `c` ⇒ ít nhất 1 viên từ `a` hoặc `b` ⇒ `k ≤ a + b`.

⇒ `k ≤ min(a + b, ⌊S/2⌋)`.

**Đạt được cận:**
- Nếu `c ≥ a + b`: ghép mỗi viên của `a` và `b` với một viên của `c` ⇒ `a + b` lượt (đủ vì `c ≥ a + b`).
- Nếu `c < a + b`: như Cách 4 — ghép `a` với `c` đúng `c − b` lần (hợp lệ vì `c − b < a`) để hai đống lớn bằng nhau `b' = c'`; sau đó luân phiên ghép từng viên của `a` lúc với đống này lúc với đống kia để chênh lệch giữa hai đống lớn `≤ 1`; khi `a` hết, ghép hai đống lớn với nhau tới khi một đống hết. Chỉ dư tối đa 1 viên ⇒ `k = ⌊S/2⌋`. ∎

**Tham lam "hai đống lớn nhất" đúng** vì nó thực hiện đúng chiến lược cân bằng trên: khi `c ≥ a + b` nó luôn lấy từ `c` và một đống nhỏ; khi không, nó giữ chênh lệch giữa các đống nhỏ dần nên chỉ dừng khi còn tối đa 1 viên.

## 11. Phân tích Time / Space Complexity

Gọi `S = a + b + c ≤ 3·10⁵`.

| Cách | Time | Space |
|---|---|---|
| 1. Vét cạn ghi nhớ | `O(a·b·c)` trạng thái | `O(a·b·c)` |
| 2. Tham lam + sort | `O(S)` | `O(1)` |
| 3. Tham lam + heap | `O(S log 3) = O(S)` | `O(1)` |
| 4. Nhảy bước | `O(1)` | `O(1)` |
| 5. Công thức | `O(1)` | `O(1)` |

In [17]:
big = (10 ** 5, 10 ** 5, 10 ** 5)
for name, f in [("sort", score_greedy_sort), ("heap", score_heap), ("jump", score_jump), ("formula", score_formula)]:
    t = time.perf_counter(); r = f(*big); dt = time.perf_counter() - t
    print(f"{name:<8} -> {r} ({dt*1000:.2f} ms)")

sort     -> 150000 (12.16 ms)
heap     -> 150000 (27.90 ms)
jump     -> 150000 (0.00 ms)
formula  -> 150000 (0.00 ms)


## 12. Bộ test

Đối chiếu mọi cách với vét cạn trên **toàn bộ** lưới `1..12` (1728 bộ), thêm ngẫu nhiên lớn (seed cố định) giữa các cách nhanh, và edge case.

In [18]:
FAST = {
    "sort": score_greedy_sort, "heap": score_heap, "jump": score_jump, "formula": score_formula,
    "SolSort": lambda a, b, c: SolutionSort().maximumScore(a, b, c),
    "SolHeap": lambda a, b, c: SolutionHeap().maximumScore(a, b, c),
    "submit": lambda a, b, c: Solution().maximumScore(a, b, c),
}
def run_fast(a, b, c):
    res = {k: f(a, b, c) for k, f in FAST.items()}
    assert len(set(res.values())) == 1, (a, b, c, res)
    return next(iter(res.values()))

for (a, b, c), exp in examples:
    assert run_fast(a, b, c) == exp
for a in range(1, 13):
    for b in range(1, 13):
        for c in range(1, 13):
            assert run_fast(a, b, c) == score_bruteforce(a, b, c), (a, b, c)
print("Lưới 1..12 khớp brute force: OK")

edge = [((1, 1, 1), 1), ((1, 1, 100000), 2), ((100000, 100000, 100000), 150000),
        ((1, 100000, 100000), 100000), ((3, 3, 3), 4), ((2, 2, 4), 4)]
for t, exp in edge:
    assert run_fast(*t) == exp, t
rng = random.Random(1753)
for _ in range(300):
    run_fast(rng.randint(1, 10 ** 5), rng.randint(1, 10 ** 5), rng.randint(1, 10 ** 5))
print("Edge case + 300 test ngẫu nhiên lớn: OK")

Lưới 1..12 khớp brute force: OK


Edge case + 300 test ngẫu nhiên lớn: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Chỉ dùng `(a + b + c) // 2`

In [19]:
def wrong_half(a, b, c):
    return (a + b + c) // 2
print("sai :", wrong_half(1, 2, 10), "| đúng:", Solution().maximumScore(1, 2, 10))

sai : 6 | đúng: 3


**Vì sao sai:** khi một đống áp đảo, các viên dư của nó không có bạn ghép. **Sửa:** `min(a + b, S // 2)` với `c` là đống lớn nhất.

### Lỗi 2 — Quên sắp xếp trước khi dùng công thức

In [20]:
def wrong_no_sort(a, b, c):
    return min(a + b, (a + b + c) // 2)       # giả định c lớn nhất mà không sắp
print("sai :", wrong_no_sort(10, 1, 2), "| đúng:", Solution().maximumScore(10, 1, 2))

sai : 6 | đúng: 3


**Vì sao sai:** `a + b` chỉ là cận đúng khi `c` là **lớn nhất**. Với `(10, 1, 2)`, `a + b = 11` không phải cận. **Sửa:** `a, b, c = sorted((a, b, c))`.

### Lỗi 3 — Tham lam lấy từ hai đống **nhỏ nhất**

In [21]:
def wrong_two_smallest(a, b, c):
    s = sorted([a, b, c]); ans = 0
    while s[1] > 0:
        i, j = (0, 1) if s[0] > 0 else (1, 2)     # SAI: ưu tiên hai đống nhỏ
        s[i] -= 1; s[j] -= 1; ans += 1
        s.sort()
    return ans
print("sai :", wrong_two_smallest(4, 4, 6), "| đúng:", Solution().maximumScore(4, 4, 6))

sai : 4 | đúng: 7


**Vì sao sai:** lấy từ hai đống nhỏ làm chúng cạn sớm, đống lớn còn nhiều sỏi nhưng không còn bạn ghép. **Sửa:** luôn lấy từ **hai đống lớn nhất**.

### Lỗi 4 — Điều kiện dừng sai: dừng khi **có một** đống rỗng

In [22]:
def wrong_stop(a, b, c):
    s = sorted([a, b, c]); ans = 0
    while s[0] > 0:                                 # SAI: phải là còn >= 2 đống không rỗng
        s[1] -= 1; s[2] -= 1; ans += 1
        s.sort()
    return ans
print("sai :", wrong_stop(2, 4, 6), "| đúng:", Solution().maximumScore(2, 4, 6))

sai : 5 | đúng: 6


**Vì sao sai:** trò chơi chỉ dừng khi còn **ít hơn hai** đống không rỗng; một đống rỗng vẫn chơi tiếp với hai đống kia. **Sửa:** `while s[1] > 0`.

### Lỗi 5 — Vét cạn đệ quy không ghi nhớ (quá chậm)

In [23]:
def slow_no_memo(a, b, c, counter):
    counter[0] += 1
    res = 0
    for nx in moves((a, b, c)):
        res = max(res, 1 + slow_no_memo(*nx, counter))
    return res

cnt = [0]
t = time.perf_counter(); r = slow_no_memo(3, 3, 4, cnt); dt = time.perf_counter() - t
print(f"không ghi nhớ: kết quả {r}, {cnt[0]} lần gọi, {dt*1000:.1f} ms | có ghi nhớ: {score_bruteforce(3, 3, 4)}")

không ghi nhớ: kết quả 5, 120 lần gọi, 0.1 ms | có ghi nhớ: 5


**Vì sao sai (về hiệu năng):** cùng một trạng thái bị tính lại rất nhiều lần — số lần gọi tăng theo hàm mũ. **Sửa:** `lru_cache` + chuẩn hóa trạng thái (hoặc dùng công thức).

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Mỗi bước lấy 1 từ **hai** nhóm khác nhau" ⇒ hai cận: `⌊tổng/2⌋` và `tổng − max`.
- Bài game nhỏ ⇒ viết vét cạn ghi nhớ trước để **đoán và kiểm chứng** công thức.
- Ghép cặp giữa các nhóm ⇒ tham lam "hai nhóm lớn nhất" bằng heap (tổng quát cho `k` nhóm).
- Với `k` nhóm tổng quát: đáp án `min(⌊S/2⌋, S − max)` — cùng lập luận.

## 15. Biến thể của bài

1. **k đống** (thay vì 3): công thức `min(S // 2, S − max)` vẫn đúng; tham lam heap vẫn chạy — Bài tập 5.
2. **Mỗi lượt lấy từ ba đống**: điểm = `min(a, b, c)`.
3. **1953 Maximum Number of Weeks for Which You Can Work** — cùng cấu trúc: không làm hai tuần liên tiếp cùng dự án ⇒ `min(S, 2·(S − max) + 1)`.
4. **767 Reorganize String / 1054 Distant Barcodes** — heap hai phần tử lớn nhất xen kẽ.
5. **Trả về chuỗi nước đi** cụ thể (Bài tập 4).

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vét cạn ghi nhớ | Thử mọi nước đi | `O(abc)` | `O(abc)` | Kiểm chứng, giá trị nhỏ |
| 2. Tham lam + sort | Lấy hai đống lớn nhất | `O(S)` | `O(1)` | Dễ viết, dễ thuyết phục |
| 3. Tham lam + heap | Như 2, tổng quát k đống | `O(S log k)` | `O(k)` | Nhiều đống |
| 4. Nhảy bước | Cân bằng rồi gộp | `O(1)` | `O(1)` | Hiểu vì sao công thức đúng |
| 5. Công thức | `min(a+b, S//2)` | `O(1)` | `O(1)` | **Bản nộp khuyên dùng** |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** (a) `(1,1,1)`; (b) `(5,5,5)`; (c) `(1,1,7)`; (d) `(6,7,8)`.

**Bài 2 (tính tay):** Với `(3, 5, 7)` tính hai cận trên và đáp án; trace 4 lượt đầu của tham lam.

**Bài 3 (điền TODO):** Hoàn thiện `upper_bounds(a, b, c)` trả `(⌊S/2⌋, S − max)`.

**Bài 4 (tự viết lại):** Viết `play(a, b, c)` trả về **danh sách nước đi** dạng cặp chỉ số đống `(i, j)` (0-based) theo tham lam; kiểm tra danh sách hợp lệ và có độ dài bằng đáp án.

**Bài 5 (mini challenge):** Viết `max_score_k(piles)` cho **k ≥ 2** đống bằng công thức, đối chiếu tham lam heap và vét cạn nhỏ.

In [24]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def upper_bounds(a, b, c):
    # TODO: S = a + b + c; trả (S // 2, S - max(a, b, c))
    pass

In [25]:
# Bài 4, 5 — khung tự viết (chỉ định nghĩa, không gọi)
def play(a, b, c):
    # TODO: mỗi lượt chọn hai chỉ số có giá trị lớn nhất (giá trị > 0), ghi lại cặp
    pass

def max_score_k(piles):
    # TODO: S = sum(piles); trả min(S // 2, S - max(piles))
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** Mức 1: sắp xếp. Mức 2: so `c` với `a + b`. Mức 3: (a) `1`; (b) `7`; (c) `2`; (d) `10`.

**Bài 2.** Mức 1: `S = 15`. Mức 2: cận `7` và `8`. Mức 3: đáp án `7`; 4 lượt đầu: [3,5,7] → [3,4,6] → [3,3,5] → [2,3,4] (sau khi sắp) → [2,2,3].

**Bài 3.** Mức 1: tính `S`. Mức 2: `max(a, b, c)`. Mức 3: trả tuple.

**Bài 4.** Mức 1: giữ list `v = [a, b, c]`. Mức 2: `i, j = sorted(range(3), key=lambda t: -v[t])[:2]`. Mức 3: dừng khi `v[j] == 0`; kiểm tra bằng cách áp dụng lại các nước đi và xem không đống nào âm.

**Bài 5.** Mức 1: hai cận như bài gốc. Mức 2: cận `S − max` vì mỗi lượt có ≥ 1 viên ngoài đống lớn nhất. Mức 3: tham lam heap đạt cận; vét cạn với `k = 4`, giá trị `≤ 4` để kiểm.

## 19. Đáp án tham khảo cho bài tập

In [26]:
# Bài 1, 2, 3
b1 = [Solution().maximumScore(*t) for t in [(1, 1, 1), (5, 5, 5), (1, 1, 7), (6, 7, 8)]]
print("Bài 1:", b1); assert b1 == [1, 7, 2, 10]

def upper_bounds(a, b, c):
    S = a + b + c
    return S // 2, S - max(a, b, c)
print("Bài 2/3: cận =", upper_bounds(3, 5, 7)); assert upper_bounds(3, 5, 7) == (7, 8)
assert trace_greedy(3, 5, 7) == 7

Bài 1: [1, 7, 2, 10]
Bài 2/3: cận = (7, 8)
bắt đầu: [3, 5, 7] | công thức dự đoán: 7
lượt 1: [3, 5, 7] -> lấy 5 & 7 -> [3, 4, 6]
lượt 2: [3, 4, 6] -> lấy 4 & 6 -> [3, 3, 5]
lượt 3: [3, 3, 5] -> lấy 3 & 5 -> [2, 3, 4]
lượt 4: [2, 3, 4] -> lấy 3 & 4 -> [2, 2, 3]
lượt 5: [2, 2, 3] -> lấy 2 & 3 -> [1, 2, 2]
lượt 6: [1, 2, 2] -> lấy 2 & 2 -> [1, 1, 1]
lượt 7: [1, 1, 1] -> lấy 1 & 1 -> [0, 0, 1]
Kết quả: 7 | sỏi còn dư: 1


In [27]:
# Bài 4
def play(a, b, c):
    v = [a, b, c]; seq = []
    while True:
        i, j = sorted(range(3), key=lambda t: -v[t])[:2]
        if v[j] == 0:
            return seq
        v[i] -= 1; v[j] -= 1; seq.append((i, j))

def valid(a, b, c, seq):
    v = [a, b, c]
    for i, j in seq:
        if i == j or v[i] == 0 or v[j] == 0:
            return False
        v[i] -= 1; v[j] -= 1
    return sum(1 for x in v if x > 0) < 2          # trò chơi đã kết thúc

for a in range(1, 9):
    for b in range(1, 9):
        for c in range(1, 9):
            s = play(a, b, c)
            assert valid(a, b, c, s) and len(s) == Solution().maximumScore(a, b, c)
print("Bài 4: OK | ví dụ (2,4,6):", play(2, 4, 6))

Bài 4: OK | ví dụ (2,4,6): [(2, 1), (2, 1), (2, 0), (2, 1), (2, 0), (1, 2)]


In [28]:
# Bài 5
def max_score_k(piles):
    S = sum(piles)
    return min(S // 2, S - max(piles))

def heap_k(piles):
    h = [-x for x in piles if x > 0]; heapq.heapify(h); ans = 0
    while len(h) >= 2:
        x, y = -heapq.heappop(h), -heapq.heappop(h); ans += 1
        if x > 1: heapq.heappush(h, -(x - 1))
        if y > 1: heapq.heappush(h, -(y - 1))
    return ans

@lru_cache(maxsize=None)
def brute_k(state):
    best = 0
    k = len(state)
    for i in range(k):
        for j in range(i + 1, k):
            if state[i] and state[j]:
                nx = list(state); nx[i] -= 1; nx[j] -= 1
                best = max(best, 1 + brute_k(tuple(sorted(nx))))
    return best

rng = random.Random(5)
for _ in range(300):
    k = rng.randint(2, 4)
    piles = [rng.randint(0, 4) for _ in range(k)]
    assert max_score_k(piles) == heap_k(piles) == brute_k(tuple(sorted(piles))), piles
print("Bài 5: OK | ví dụ [1,1,1,9]:", max_score_k([1, 1, 1, 9]), "| [3,3,3,3]:", max_score_k([3, 3, 3, 3]))

Bài 5: OK | ví dụ [1,1,1,9]: 3 | [3,3,3,3]: 6


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template:**

```python
a, b, c = sorted((a, b, c))
return a + b if c >= a + b else (a + b + c) // 2
# tổng quát k đống: min(S // 2, S - max(piles))
```

**Checklist:**
- [ ] Đã **sắp xếp** để `c` là đống lớn nhất?
- [ ] Xét cả hai cận: `⌊S/2⌋` **và** `a + b`?
- [ ] Nếu mô phỏng: lấy từ **hai đống lớn nhất**, dừng khi đống lớn nhì bằng 0?
- [ ] Đã test: ba số bằng nhau, một đống áp đảo, `(1,1,1)`, giá trị `10⁵`?